In [1]:
%pip install -qU google-generativeai
%pip install -qU google-ai-generativelanguage==0.6.15
%pip install -qU langchain-google-genai
%pip install -qU langchain-community
%pip install -qU langchain
%pip install -qU langgraph
%pip install -qU langgraph langchain-community
%pip install -qU python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


https://react-lm.github.io/

https://smith.langchain.com/hub/langchain-ai/react-agent-template

In [2]:
import os
import re
import google.genai as genai
from langgraph.graph import StateGraph, END
from typing import TypedDict

In [5]:
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()

GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel('gemini-flash-latest')
response = model.generate_content("Hello world")

print(response.text)

Hello, World! 👋 

How can I help you today?


In [6]:
from dotenv import load_dotenv

In [7]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})
    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute(message)
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self, message):
        client = genai.GenerativeModel("gemini-flash-latest")
        completion = client.generate_content(message)
        return completion.text

In [8]:
agente = Agent(system="Eres un asistente útil y servicial que responde a las preguntas de los usuarios de manera clara y concisa.")
print(agente("¿Cuál es la capital de Francia?"))

La capital de Francia es **París**.


## 4. Agentes y Funciones

In [9]:
PROMPT_REACT = """
Funcionas en un ciclo de Pensamiento, Acción, Pausa y Observación.
Al final del ciclo, proporcionas una Respuesta.
Usa "Pensamiento" para describir tu razonamiento.
Usa "Acción" para ejecutar herramientas - y luego retorna "PAUSA".
La "Observación" será el resultado de la acción ejecutada.
Acciones disponibles:

consultar_stock: devuelve la cantidad disponible de un articulo en el inventario (ej: "consultar_stock: teclado")

consultar_precio_producto: devuelve el precio unitario de un producto (ej: "consultar_precio_producto: mouse gamer")

Ejemplo:
Pregunta: ¿Cuántos monitores tenemos en el inventario?
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de monitores.
Acción: consultar_stock: monitor
PAUSA

Observación: Tenemos 75 monitores en el inventario.
Respuesta: Hay 75 monitores en el inventario.
""".strip()

In [10]:
class EstadoAgente(TypedDict):
    pregunta: str
    historial: list[str]
    accion_pendiente: str
    respuesta_final: str

In [11]:
def consultar_stock(item: str) -> str:
    """Simula la consulta de stock de item en el inventario."""

    item = item.lower()
    stock = {
        "monitor": 75,
        "teclado": 120,
        "mouse de gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impresora": 15
    }

    if item in stock:
        return f"Tenemos {stock[item]} {item}s en stock."
    else:
        return f"Item '{item}' no encontrado en el inventario."

def consultar_precio_producto(producto: str) -> str:
    """Simula la consulta del precio unitario de un producto."""
    producto = producto.lower()
    precios = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse de gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impresora": 750.00
    }

    if producto in precios:
        return f"El precio de un(a) {producto} es USD {precios[producto]:.2f}."
    else:
        return f"producto '{producto}' no hallado en la lista de precios."

In [12]:
print(consultar_stock("teclado"))
print(consultar_precio_producto("impresora"))
print(consultar_stock("monitor"))

Tenemos 120 teclados en stock.
El precio de un(a) impresora es USD 750.00.
Tenemos 75 monitors en stock.


In [13]:
print(consultar_stock("sillas"))

Item 'sillas' no encontrado en el inventario.


## 05 Nuevas Funciones

In [20]:
def run_react_agent(pregunta: str, max_iterations: int = 5) -> str:
    model = genai.GenerativeModel('gemini-flash-latest')
    chat = model.start_chat(history=[])

    chat.send_message(PROMPT_REACT)
    
    current_prompt = pregunta

    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"--- Iteración {i+1} ---")
        print(f"Modelo pensó/respondió:\n{response_text}\n")

        if response_text.startswith("Respuesta:"):
            return response_text.replace("Respuesta:", "").strip()

        match = re.search(r"Acción:\s*(\w+)\s*\((.*)\)", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip()

            observacion = ""
            if action_name == "consultar_stock":
                observacion = consultar_stock(action_arg)
            elif action_name == "consultar_precio_producto":
                observacion = consultar_precio_producto(action_arg)
            else:
                observacion = f"Error: Acción '{action_name}' desconocida."
                current_prompt = f"Observación: {observacion}\nRespuesta:"
            
                print(f"Ejecutó acción: {action_name}({action_arg})")
                print(f"Observación: {observacion}\n")
        else:
            return f"Error: El agente no logró extraer una Acción o Respuesta final tras {i+1} iteraciones. La última respuesta fue: {response_text}"

    return "Error: Límite máximo de iteraciones alcanzado sin una respuesta final."

In [21]:
pregunta_1 = "Cuántos mouses de gamer están disponibles en el inventario?"
print(f"**Interacción 1: {pregunta_1}**")
respuesta_1 = run_react_agent(pregunta_1)
print(f"\n**RESPUESTA FINAL DEL AGENTE 1:** {respuesta_1}\n")

print("\n" + "="*50 + "\n")

**Interacción 1: Cuántos mouses de gamer están disponibles en el inventario?**
--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Necesito saber la cantidad disponible de mouses gamer en el inventario, por lo que debo usar la acción `consultar_stock`.
Acción: consultar_stock: mouse gamer
PAUSA


**RESPUESTA FINAL DEL AGENTE 1:** Error: El agente no logró extraer una Acción o Respuesta final tras 1 iteraciones. La última respuesta fue: Pensamiento: Necesito saber la cantidad disponible de mouses gamer en el inventario, por lo que debo usar la acción `consultar_stock`.
Acción: consultar_stock: mouse gamer
PAUSA





In [22]:
pregunta_1 = "Cuántos mouse de gamer están disponibles en el inventario?"

In [23]:
pregunta_2 = "Cuál es el precio de una impresora?"
print(f"**Interacción 2: {pregunta_2}**")
respuesta_2 = run_react_agent(pregunta_2)
print(f"\n**RESPUESTA FINAL DEL AGENTE 2:** {respuesta_2}\n")

print("\n" + "="*50 + "\n")

**Interacción 2: Cuál es el precio de una impresora?**
--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Debo consultar el precio de la impresora utilizando la herramienta disponible.
Acción: consultar_precio_producto: impresora
PAUSA


**RESPUESTA FINAL DEL AGENTE 2:** Error: El agente no logró extraer una Acción o Respuesta final tras 1 iteraciones. La última respuesta fue: Pensamiento: Debo consultar el precio de la impresora utilizando la herramienta disponible.
Acción: consultar_precio_producto: impresora
PAUSA





In [24]:
pregunta_3 = "Tenemos sillas en inventario?"
print(f"**Interacción 3: {pregunta_3}**")
respuesta_3 = run_react_agent(pregunta_3)
print(f"\n**RESPUESTA FINAL DEL AGENTE 3:** {respuesta_3}\n")

**Interacción 3: Tenemos sillas en inventario?**
--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: El usuario quiere saber si hay sillas en el inventario. Debo utilizar la herramienta `consultar_stock` para verificar la cantidad disponible de sillas.
Acción: consultar_stock: silla
PAUSA


**RESPUESTA FINAL DEL AGENTE 3:** Error: El agente no logró extraer una Acción o Respuesta final tras 1 iteraciones. La última respuesta fue: Pensamiento: El usuario quiere saber si hay sillas en el inventario. Debo utilizar la herramienta `consultar_stock` para verificar la cantidad disponible de sillas.
Acción: consultar_stock: silla
PAUSA



In [25]:
pregunta_4 = "Cuál es el producto más costoso?"
print(f"**Interacción 4: {pregunta_4}**")
respuesta_4 = run_react_agent(pregunta_4)
print(f"\n**RESPUESTA FINAL DEL AGENTE 4:** {respuesta_4}\n")

**Interacción 4: Cuál es el producto más costoso?**
--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Las herramientas actuales solo permiten consultar precios de productos individuales y no proporcionan un listado general ni una búsqueda del valor máximo en el catálogo.

Respuesta: No cuento con una acción para consultar directamente el producto más costoso o listar todo el catálogo. Si me indicas los productos específicos que deseas comparar, puedo consultar el precio de cada uno para determinar cuál es el más caro.


**RESPUESTA FINAL DEL AGENTE 4:** Error: El agente no logró extraer una Acción o Respuesta final tras 1 iteraciones. La última respuesta fue: Pensamiento: Las herramientas actuales solo permiten consultar precios de productos individuales y no proporcionan un listado general ni una búsqueda del valor máximo en el catálogo.

Respuesta: No cuento con una acción para consultar directamente el producto más costoso o listar todo el catálogo. Si me indicas los producto

In [26]:
def herramienta_encontrar_producto_mas_costoso() -> str:
    """
    Retorna el nombre y el precio del producto más costoso en el inventario.
    Esta función no requiere argumentos adicionales.
    """
    precios_del_inventario = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse de gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impresora": 750.00
    }
    
    if not precios_del_inventario:
        return "Lo sentimos, no hallamos ningún producto en la lista de precios para su comparación."

    nombre_producto_mas_costoso = max(precios_del_inventario, key=precios_del_inventario.get)
    valor_producto_mas_costoso = precios_del_inventario[nombre_producto_mas_costoso]
    
    return f"El producto más costoso es el(la) {nombre_producto_mas_costoso} con precio de USD {valor_producto_mas_costoso:.2f}."

## 06 Agente ReAct Interactivo

In [27]:
PROMPT_REACT = """
Funciona en un ciclo de Pensamiento, Acción, Pausa y Observación.
Al final del ciclo, proporcionas una Respuesta.
Usa "Pensamiento" para describir tu razonamiento.
Usa "Acción" para ejecutar herramientas, y luego regresa "PAUSA".
La "Observación" será el resultado de la acción ejecutada.
Acciones disponibles:
- consultar_stock: devuelve la cantidad disponible de un artículo en el inventario (ej.: "consultar_stock: teclado")
- consultar_precio_producto: devuelve el precio unitario de un producto (ej.: "consultar_precio_producto: mouse gamer")
- encontrar_producto_mas_costoso: devuelve el nombre y el precio del producto más costoso del inventario (no requiere argumentos)

Ejemplo:
Pregunta: ¿Cuántos monitores tenemos en stock?
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de monitores.
Acción: consultar_stock: monitor
PAUSA

Observación: Tenemos 75 monitores en stock.
Respuesta: Hay 75 monitores en stock.

Ejemplo:
Pregunta: ¿Cuál es el producto más costoso?
Pensamiento: Necesito usar la acción encontrar_producto_mas_costoso para descubrir qué producto tiene el precio más alto.
Acción: encontrar_producto_mas_costoso
PAUSA

Observación: El producto más costoso es el monitor con un precio de R$ 999,90.
Respuesta: El producto más costoso es el monitor con un precio de R$ 999,90.
""".strip()

In [28]:
def run_react_agent(pregunta: str, max_iterations: int = 5) -> str:
    """
    Ejecuta el ciclo ReAct para una determinada pregunta usando el modelo Gemini.
    """

    model = genai.GenerativeModel('gemini-1.5-flash')
    chat = model.start_chat(history=[])
    chat.send_message(PROMPT_REACT)

    current_prompt = pregunta
    
    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"\n--- Iteración {i+1} ---")
        print(f"Modelo pensó/respondió:\n{response_text}\n")
        
        response_match_final = re.search(r"Respuesta:\s*(.*)", response_text, re.DOTALL)
        if response_match_final:
            return response_match_final.group(1).strip()

        match = re.search(r"Acción:\s*(\w+)(?::\s*(.*))?", response_text)
        
        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""
            observacion_de_accion = ""
            
            if action_name == "consultar_stock":
                observacion_de_accion = consultar_stock(action_arg)
            elif action_name == "consultar_precio_producto":
                observacion_de_accion = consultar_precio_producto(action_arg)
            elif action_name == "encontrar_producto_mas_costoso":
                observacion_de_accion = herramienta_encontrar_producto_mas_costoso()
            else:
                observacion_de_accion = f"Error: Acción '{action_name}' desconocida. Verifica el prompt o la implementación de la herramienta."
            
            # Simplificamos el prompt de Observación
            current_prompt = f"Observación: {observacion_de_accion}"
            
            print(f"Ejecutó acción: {action_name} con argumento '{action_arg}'")
            print(f"Observación: {observacion_de_accion}\n")

        else:
            return f"Error: El agente no logró extraer una Acción o Respuesta final tras {i+1} iteraciones. La última respuesta fue: '{response_text}'"

    return f"Error: Límite máximo de iteraciones alcanzado sin una respuesta final."

In [29]:
def herramienta_calculador_valor_total_lista(lista_items: str) -> str:
    """
    Calcula el valor total de una lista de items de compra.
    Recibe una string con items separados por coma (ex: "teclado, mouse de gamer, monitor").
    """
    
    precios_del_inventario = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse de gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impresora": 750.00
    }
    
    items_procesados = [item.strip().lower() for item in lista_items.split(',')]
    
    valor_total = 0.0
    items_no_encontrados = []
    
    for item in items_procesados:
        if item in precios_del_inventario:
            valor_total += precios_del_inventario[item]
        else:
            items_no_encontrados.append(item)
            
    respuesta = f"El valor total de los items encontrados es USD {valor_total:.2f}."
    if items_no_encontrados:
        respuesta += f"Los siguientes items no fueron encontrados y tampoco incluidos en el cálculo: {', '.join(items_no_encontrados)}."
        
    return respuesta

In [31]:
print("Testeando la herramienta_calculador_valor_total_lista")
lista_1 = "teclado, mouse de gamer, monitor"
resultado_1 = herramienta_calculador_valor_total_lista(lista_1)
print(f"Lista: {lista_1}\nResultado: {resultado_1}\n")

Testeando la herramienta_calculador_valor_total_lista
Lista: teclado, mouse de gamer, monitor
Resultado: El valor total de los items encontrados es USD 1249.40.



In [33]:
lista_2 = "headset, silla"
resultado_2 = herramienta_calculador_valor_total_lista(lista_2)

print(f"Lista: '{lista_2}'\nResultado: {resultado_2}\n")

Lista: 'headset, silla'
Resultado: El valor total de los items encontrados es USD 180.00.Los siguientes items no fueron encontrados y tampoco incluidos en el cálculo: silla.



In [36]:
lista_3 = "mesa, vaso"
resultado_3 = herramienta_calculador_valor_total_lista(lista_3)
print(f"Lista: '{lista_3}'\nResultado: {resultado_3}\n")

Lista: 'mesa, vaso'
Resultado: El valor total de los items encontrados es USD 0.00.Los siguientes items no fueron encontrados y tampoco incluidos en el cálculo: mesa, vaso.



In [54]:
def run_react_agent(pregunta: str, max_iterations: int = 5) -> str:
    """
    Ejecuta el ciclo ReAct para una determinada pregunta usando el modelo Gemini.
    """

    model = genai.GenerativeModel('gemini-2.5-flash')
    chat = model.start_chat(history=[])
    chat.send_message(PROMPT_REACT)

    current_prompt = pregunta
    
    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"\n--- Iteración {i+1} ---")
        print(f"Modelo pensó/respondió:\n{response_text}\n")
        
        response_match_final = re.search(r"Respuesta:\s*(.*)", response_text, re.DOTALL)
        if response_match_final:
            return response_match_final.group(1).strip()

        match = re.search(r"Acción:\s*(\w+)(?::\s*(.*))?", response_text)
        
        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""
            observacion_de_accion = ""
            
            if action_name == "consultar_stock":
                observacion_de_accion = consultar_stock(action_arg)
            elif action_name == "consultar_precio_producto":
                observacion_de_accion = consultar_precio_producto(action_arg)
            elif action_name == "encontrar_producto_mas_costoso":
                observacion_de_accion = herramienta_encontrar_producto_mas_costoso()
            elif action_name == "calculador_valor_total_lista":
                observacion_de_accion = herramienta_calculador_valor_total_lista(action_arg)
            else:
                observacion_de_accion = f"Error: Acción '{action_name}' desconocida. Verifica el prompt o la implementación de la herramienta."
            
            # Simplificamos el prompt de Observación
            current_prompt = f"Observación: {observacion_de_accion}"
            
            print(f"Ejecutó acción: {action_name} con argumento '{action_arg}'")
            print(f"Observación: {observacion_de_accion}\n")

        else:
            return f"Error: El agente no logró extraer una Acción o Respuesta final tras {i+1} iteraciones. La última respuesta fue: '{response_text}'"

    return f"Error: Límite máximo de iteraciones alcanzado sin una respuesta final."

In [55]:
PROMPT_REACT = """
Funciona en un ciclo de Pensamiento, Acción, Pausa y Observación.
Al final del ciclo, proporcionas una Respuesta.
Usa "Pensamiento" para describir tu razonamiento.
Usa "Acción" para ejecutar herramientas, y luego regresa "PAUSA".
La "Observación" será el resultado de la acción ejecutada.
Acciones disponibles:
- consultar_stock: devuelve la cantidad disponible de un artículo en el inventario (ej.: "consultar_stock: teclado")
- consultar_precio_producto: devuelve el precio unitario de un producto (ej.: "consultar_precio_producto: mouse gamer")
- encontrar_producto_mas_costoso: devuelve el nombre y el precio del producto más costoso del inventario (no requiere argumentos)
- calculador_valor_total_lista: calcula el valor total de una lista de items de compra (ej.: "calculador_valor_total_lista: teclado, mouse de gamer, monitor")

Ejemplo:
Pregunta: ¿Cuántos monitores tenemos en stock?
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de monitores.
Acción: consultar_stock: monitor
PAUSA

Observación: Tenemos 75 monitores en stock.
Respuesta: Hay 75 monitores en stock.

Ejemplo:
Pregunta: ¿Cuál es el producto más costoso?
Pensamiento: Necesito usar la acción encontrar_producto_mas_costoso para descubrir qué producto tiene el precio más alto.
Acción: encontrar_producto_mas_costoso
PAUSA

Observación: El producto más costoso es el monitor con un precio de R$ 999,90.
Respuesta: El producto más costoso es el monitor con un precio de R$ 999,90.
""".strip()

In [56]:
print("--- Iniciando a interactuar con el Agente ReAct ---")

# Interacción 1: Consultar Stock
pregunta_1 = "Cuántos teclados tenemos en stock?"
print(f"\n**Interacción 1: {pregunta_1}**")
respuesta_1 = run_react_agent(pregunta_1)
print(f"\n**RESPUESTA FINAL DEL AGENTE 1:** {respuesta_1}\n")

print("\n" + "="*80 + "\n")

# Interacción 2: Consultar Precio
pregunta_2 = "Cuál es el precio de un headset?"
print(f"\n**Interacción 2: {pregunta_2}**")
respuesta_2 = run_react_agent(pregunta_2)
print(f"\n**RESPUESTA FINAL DEL AGENTE 2:** {respuesta_2}\n")

print("\n" + "="*80 + "\n")

# Interacción 3: Item no encontrado en stock
pregunta_3 = "Tenemos sillas en stock?"
print(f"\n**Interacción 3: {pregunta_3}**")
respuesta_3 = run_react_agent(pregunta_3)
print(f"\n**RESPUESTA FINAL DEL AGENTE 3:** {respuesta_3}\n")

print("\n" + "="*80 + "\n")

# Interacción 4: Encontrar el Produto Más Costoso
pregunta_4 = "Cuál es el producto más costoso?"
print(f"\n**Interacción 4: {pregunta_4}**")
respuesta_4 = run_react_agent(pregunta_4)
print(f"\n**RESPUESTA FINAL DEL AGENTE 4:** {respuesta_4}\n")

print("\n" + "="*80 + "\n")

# Interacción 5: Calcular el Valor Total de la Lista (NUEVA FUNCIONALIDAD)
pregunta_5 = "Cuál es el valor de un teclado, una impresora y una webcam?"
print(f"\n**Interacción 5: {pregunta_5}**")
respuesta_5 = run_react_agent(pregunta_5)
print(f"\n**RESPUESTA FINAL DEL AGENTE 5:** {respuesta_5}\n")

print("\n--- Fin de las Interacciones ---")

--- Iniciando a interactuar con el Agente ReAct ---

**Interacción 1: Cuántos teclados tenemos en stock?**


NotFound: 404 This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements.

In [57]:
def iniciar_conversacion_con_agente():
    print("--- Agente de Inventario Interactivo ---")
    print("Realiza tu pregunta sobre el inventario, o digita 'salir' para cerrar la sesión.")
    print("-" * 50)
    
    while True:
        pregunta_usuario = input("\nVocê: ")
        
        if pregunta_usuario.lower().strip() == 'salir':
            print("¡Atención finalizada. Hasta pronto!")
            break
            
        print("\nAgente: Procesando...")
        try:
            respuesta_agente = run_react_agent(pregunta_usuario)
            print(f"\nAgente: {respuesta_agente}")
        except Exception as e:
            print(f"\nAgente: Ocurrió un error al procesar su pregunta: {e}")
            print("Por favor, intenta nuevamente, o digita 'salir'.")

In [ ]:
iniciar_conversacion_con_agente()


--- Agente de Inventario Interactivo ---
Realiza tu pregunta sobre el inventario, o digita 'salir' para cerrar la sesión.
--------------------------------------------------

Agente: Procesando...

Agente: Ocurrió un error al procesar su pregunta: 404 This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements.
Por favor, intenta nuevamente, o digita 'salir'.

Agente: Procesando...

Agente: Ocurrió un error al procesar su pregunta: 404 This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements.
Por favor, intenta nuevamente, o digita 'salir'.

Agente: Procesando...

Agente: Ocurrió un error al procesar su pregunta: 404 This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and im